In [ ]:
# ============================================================
# NB_TRNSF_SilverADVWDB_Product
# Capa: Silver | Origen: ADVWDB | Tabla: Product
# Carga TOTAL — overwrite en cada ejecución
# FLUJO: CopyData (SQL→lh_silver staging) → este NB → lh_silver final
# FIX: el CopyData deposita en lh_silver.<origen>.<tabla>_staging
# Este notebook lee ese staging, limpia, tipa y escribe Silver final
# Todo desde config — sin hardcodeos
# ============================================================
from pyspark.sql import functions as F

# ============================================================
# IDs de workspace y lakehouses — SIN hardcodear (TP)
# ============================================================
_ctx    = notebookutils.runtime.context
WS_ID   = _ctx['workspaceId']
_lh_map = {lh.displayName: lh.id for lh in notebookutils.lakehouse.list()}
ART_SILVER = _lh_map.get('lh_silver', '')


# ── IDs técnicos (única excepción documentada en TP) ─────────

# ── Config desde lh_config ───────────────────────────────────
config = spark.sql("""
    SELECT *
    FROM lh_config.dbo.origin_bronze_silver
    WHERE origen       = 'ADVWDB'
      AND nombre_tabla = 'Product'
      AND activo       = 1
    LIMIT 1
""").collect()[0]

nombre_tabla = config['nombre_tabla']         # Product
origen       = config['origen']               # ADVWDB
pks          = [pk.strip() for pk in config['pks'].split(',')]
tipo_carga   = config['tipo_carga']           # total

# Nombres de tabla construidos dinámicamente
tabla_staging = f'lh_silver.{origen}.{nombre_tabla}_staging'
tabla_silver  = f'lh_silver.{origen}.{nombre_tabla}'

print('✅ Config cargada')
print(f'   nombre_tabla  : {nombre_tabla}')
print(f'   origen        : {origen}')
print(f'   pks           : {pks}')
print(f'   tipo_carga    : {tipo_carga}')
print(f'   tabla_staging : {tabla_staging}')
print(f'   tabla_silver  : {tabla_silver}')


In [ ]:
# ============================================================
# Celda 2 — Leer desde STAGING + diagnóstico detallado
# FIX: si staging tiene 0 filas, muestra el schema para
# confirmar que cp_Product escribió correctamente.
# LIMIT 1000 obligatorio durante desarrollo (TP)
# ============================================================

try:
    df_raw = spark.sql(f'SELECT * FROM {tabla_staging} LIMIT 1000')
    filas_staging = df_raw.count()
    print(f'✅ Staging encontrado: {tabla_staging}')
    print(f'   Filas en staging   : {filas_staging}')
    print(f'   Columnas           : {df_raw.columns}')
    display(df_raw.limit(5))
except Exception as e:
    print(f'❌ Staging no encontrado: {tabla_staging}')
    print(f'   Error: {e}')
    print()
    print('DIAGNÓSTICO: el staging no existe, posibles causas:')
    print('  1. cp_Product en el pipeline usó pipeline().parameters (bug)')
    print('     → verificar que DP_INGST_SilverADVWDB usa filter_ADVWDB.output.value[1].*')
    print('  2. El Copy Data escribió en un schema distinto')
    print('     → revisar actividad cp_Product en el pipeline')
    raise Exception(f'Staging no encontrado. Reimportar DP_INGST_SilverADVWDB.json corregido.')

if filas_staging == 0:
    print('❌ Staging tiene 0 filas')
    print()
    print('DIAGNÓSTICO: staging existe pero vacío, posibles causas:')
    print('  1. El CopyData no encontró registros con el watermark actual')
    print('     → Revisar ultimo_valor_incremental en origin_bronze_silver')
    print('     → Verificar conexión SQL Server AdventureWorks2019')
    print()
    print('SOLUCIÓN: reimportar DP_INGST_SilverADVWDB.json con query corregida:')
    print('  @concat(\'SELECT TOP 1000 * FROM \',')
    print('  activity(\'filter_ADVWDB\').output.value[1].ubicacion_relativa,\'.\',')
    print('  activity(\'filter_ADVWDB\').output.value[1].nombre_tabla,')
    print('  \' ORDER BY \',activity(\'filter_ADVWDB\').output.value[1].pks)')
    raise Exception('Staging vacío. Ver diagnóstico arriba.')


In [ ]:
# ============================================================
# Celda 3 — Tipado correcto y limpieza
# FIX: logging intermedio antes/después de dropna
# para confirmar si el cast introduce NULLs en PKs
# ============================================================

str_cols = [
    'Name', 'ProductNumber', 'Color', 'Size',
    'ProductLine', 'Class', 'Style',
    'SizeUnitMeasureCode', 'WeightUnitMeasureCode'
]

df_typed = (
    df_raw
    .withColumn('ProductID',            F.col('ProductID').cast('integer'))
    .withColumn('MakeFlag',             F.col('MakeFlag').cast('boolean'))
    .withColumn('FinishedGoodsFlag',    F.col('FinishedGoodsFlag').cast('boolean'))
    .withColumn('SafetyStockLevel',     F.col('SafetyStockLevel').cast('smallint'))
    .withColumn('ReorderPoint',         F.col('ReorderPoint').cast('smallint'))
    .withColumn('StandardCost',         F.col('StandardCost').cast('decimal(18,4)'))
    .withColumn('ListPrice',            F.col('ListPrice').cast('decimal(18,4)'))
    .withColumn('Weight',               F.col('Weight').cast('decimal(8,2)'))
    .withColumn('DaysToManufacture',    F.col('DaysToManufacture').cast('integer'))
    .withColumn('ProductSubcategoryID', F.col('ProductSubcategoryID').cast('integer'))
    .withColumn('ProductModelID',       F.col('ProductModelID').cast('integer'))
    .withColumn('SellStartDate',        F.col('SellStartDate').cast('timestamp'))
    .withColumn('SellEndDate',          F.col('SellEndDate').cast('timestamp'))
    .withColumn('DiscontinuedDate',     F.col('DiscontinuedDate').cast('timestamp'))
    .withColumn('ModifiedDate',         F.col('ModifiedDate').cast('timestamp'))
)

for c in str_cols:
    if c in df_typed.columns:
        df_typed = df_typed.withColumn(c, F.trim(F.col(c)))
        df_typed = df_typed.withColumn(
            c, F.when(F.col(c) == '', None).otherwise(F.col(c))
        )

# ── DIAGNÓSTICO: NULLs en PKs después del cast ───────────────
filas_antes = df_typed.count()
nulls_en_pk = df_typed.filter(F.col(pks[0]).isNull()).count()
print(f'📊 Filas después del cast    : {filas_antes}')
print(f'   NULLs en PK ({pks[0]}) : {nulls_en_pk}')
if nulls_en_pk > 0:
    print(f'   ⚠️  {nulls_en_pk} filas se eliminarán en dropna — verificar staging')
    display(df_typed.filter(F.col(pks[0]).isNull()).limit(5))

# PKs desde config — nunca hardcodeadas
df_silver = df_typed.dropna(subset=pks).dropDuplicates(pks)

filas_limpias = df_silver.count()
print(f'✅ Filas después de limpieza : {filas_limpias}')

if filas_limpias == 0:
    raise Exception(
        f'ERROR: 0 filas después de dropna/dropDuplicates.\n'
        f'  - Filas antes de dropna : {filas_antes}\n'
        f'  - NULLs en PK           : {nulls_en_pk}\n'
        f'  - PKs usadas            : {pks}\n'
        f'Verificar que config.pks coincide con nombre de columna en staging.'
    )

display(df_silver.select('ProductID','Name','ProductNumber','ListPrice','ModifiedDate').limit(5))


In [ ]:
# ============================================================
# Celda 4 — Guardar en Silver (carga TOTAL)
# DROP TABLE + saveAsTable: limpia metastore Y archivos Delta
# ============================================================

# Carga total: reemplazar todo
spark.sql(f'DROP TABLE IF EXISTS {tabla_silver}')
print(f'🗑️  Tabla anterior eliminada: {tabla_silver}')

(
    df_silver.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(tabla_silver)
)
print(f'✅ Silver guardado: {tabla_silver} ({filas_limpias} filas)')

# Limpiar staging
spark.sql(f'DROP TABLE IF EXISTS {tabla_staging}')
print(f'🗑️  Staging eliminado: {tabla_staging}')


In [ ]:
# ============================================================
# Celda 5 — Verificar resultado final
# ============================================================

total = spark.sql(
    f'SELECT COUNT(*) as cnt FROM {tabla_silver}'
).collect()[0]['cnt']

print(f'✅ Total filas en Silver ({tabla_silver}): {total}')

display(spark.sql(f"""
    SELECT ProductID, Name, ProductNumber,
           ListPrice, SellStartDate, ModifiedDate
    FROM {tabla_silver}
    ORDER BY ProductID
    LIMIT 10
"""))
